# F3 — Algoritmos para proporciones de ofertas y lectura de datos
**Grupo 5 · Víctor Bravo Barrera (algoritmos), Nayadeth Garrido Ibáñez (limpieza y validación POO) y Mauricio Cid (lectura)**

Este notebook contiene el razonamiento, las pruebas, los resultados y las decisiones de F3. El informe resume estos antecedentes. Las secciones 1 a 11 comparan algoritmos para calcular proporciones con las clases de lectura, limpieza y validación existentes. La sección 12 compara formas de leer el CSV.

La unidad de observación sigue siendo una oferta por ítem. No se deduplican licitaciones ni se infiere causalidad.

**Recorrido:** entorno → entrada → limpieza → validación → contrato del cálculo → alternativas → pruebas → resultados analíticos → mediciones → complejidad → decisión → rendimiento de la lectura → conclusión general.

Cada celda tiene un objetivo verificable. Las tablas de entrada, calidad, proporciones, tiempo y memoria se presentan por separado. Esta es una convención de organización del notebook: no significa que cada instrucción necesite su propia celda.

## 1. Entorno y presentación
### 1.1. Importaciones y ubicación del proyecto

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/analisis.py').exists())
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))
from src.entorno import versiones_entorno
from src.datos import COLUMNAS_ANALISIS, TIPOS_ANALISIS, ContratoEsquema, LectorCSV, sha256_archivo
from src.pipeline import LimpiadorLicitaciones
from src.validacion import ValidadorDatasetProcesado
from src.analisis import (tabla_proporciones, tabla_proporciones_iterativa,
                         tabla_proporciones_recursiva, tabla_proporciones_agrupada)

### 1.2. Funciones de presentación
Estas funciones cambian exclusivamente las etiquetas y el formato visible. Los DataFrames utilizados para pruebas y cálculos mantienen su estructura. Se definen aparte para que la configuración del entorno no mezcle lógica de presentación.

In [2]:
# Formato de presentación: no modifica las tablas utilizadas en los cálculos.
NOMBRES_GRUPO = {'TamanoProveedor': 'Tamaño del proveedor', 'TipoLicitacion': 'Tipo de licitación'}
NOMBRES_ALGORITMO = {'referencia_f2': 'Referencia F2', 'iterativa': 'Iterativa',
                    'recursiva': 'Recursiva', 'agrupada': 'Agrupada'}

def mostrar_tabla(tabla, titulo, formatos=None):
    estilo = (tabla.style.hide(axis='index')
              .format(formatos or {}, na_rep='—', decimal=',', thousands='.', escape='html')
              .set_caption(titulo)
              .set_table_styles([
                  {'selector': 'caption', 'props': [('text-align', 'left'), ('font-weight', 'bold'), ('padding', '12px 0 6px')]},
                  {'selector': 'th', 'props': [('text-align', 'left'), ('padding', '8px 10px'), ('white-space', 'normal'), ('border-bottom', '2px solid #94a3b8')]},
                  {'selector': 'td', 'props': [('padding', '7px 10px'), ('text-align', 'right'), ('vertical-align', 'top'), ('font-variant-numeric', 'tabular-nums'), ('border-bottom', '1px solid #cbd5e1')]},
                  {'selector': 'td.col0', 'props': [('text-align', 'left'), ('max-width', '320px'), ('white-space', 'normal')]},
              ]))
    display(estilo)

def mostrar_proporciones(tabla, titulo):
    etiqueta = NOMBRES_GRUPO.get(tabla.index.name, tabla.index.name or 'Grupo')
    vista = tabla.rename(index={'All': 'Total'}).rename_axis(etiqueta).reset_index()
    vista = vista.rename(columns={'Ganadora': 'Ganadoras', 'Perdedora': 'Perdedoras',
        'Sin resultado válido': 'Sin resultado válido', 'All': 'Ofertas válidas',
        '% Ganadora': 'Ganadoras (%)', '% Perdedora': 'Perdedoras (%)'})
    mostrar_tabla(vista, titulo, {'Ganadoras': '{:,.0f}', 'Perdedoras': '{:,.0f}',
        'Sin resultado válido': '{:,.0f}', 'Ofertas válidas': '{:,.0f}',
        'Ganadoras (%)': '{:.2f}', 'Perdedoras (%)': '{:.2f}'})

### 1.3. Evidencia del entorno
Las versiones permiten interpretar y reproducir la ejecución. Los tiempos almacenados tienen además su propio registro de entorno.

In [3]:
print(sys.version)
mostrar_tabla(pd.DataFrame(list(versiones_entorno().items()), columns=['Dependencia', 'Versión']), 'Versiones del entorno')

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


Dependencia,Versión
numpy,2.3.5
pandas,3.0.1
jupyterlab,4.6.4
ipykernel,7.3.0
nbformat,5.11.1
nbconvert,7.17.1
nbclient,0.11.0


## 2. Obtención y preparación con las clases existentes
### 2.1. Entrada y contrato de lectura
`LectorCSV` hereda de `LectorDatos` y aplica `ContratoEsquema`. Se verifica la huella de la copia utilizada en F2. La lectura queda fuera del tiempo de los algoritmos; su rendimiento se compara en la sección 12.

El contrato conserva los nombres en una tupla propia, incluso si recibe una lista. El llamador no puede cambiar sus columnas al modificar esa lista. El lector concreto implementa la interfaz abstracta. Este análisis lee solo las siete columnas necesarias, con tipos definidos al leer; la sección 12 justifica esa elección y comprueba que los datos limpios y las proporciones no cambian. La lectura completa sigue siendo la configuración por defecto, usada por F1 y F2.

In [4]:
ruta = raiz / 'data/raw/licitaciones_salud_marzo_2026.csv'
assert sha256_archivo(ruta) == '490d9209a10d387011d481b72b7891f26e997974ec2cf9dfc518aa4a08552232'
# Solo las columnas del análisis, con tipos definidos al leer (sección 12, D-33).
lector = LectorCSV(ContratoEsquema(('TamanoProveedor', 'TipoLicitacion', 'ResultadoOferta', 'EstadoLicitacion')),
                   columnas=COLUMNAS_ANALISIS, tipos=TIPOS_ANALISIS)
raw = lector.leer(ruta)
mostrar_tabla(pd.DataFrame([{'Archivo': ruta.name, 'Filas': len(raw), 'Columnas': len(raw.columns)}]),
              'Dataset de entrada', {'Filas': '{:,.0f}', 'Columnas': '{:.0f}'})

Archivo,Filas,Columnas
licitaciones_salud_marzo_2026.csv,44.226,7


### 2.2. Limpieza y conservación de filas
El limpiador encapsula la configuración y un resumen de la última ejecución. Se reutiliza la lógica de F2. Esta celda muestra solamente la transformación estructural; las reglas de calidad aparecen después.

In [5]:
limpiador = LimpiadorLicitaciones()
datos = limpiador.limpiar(raw)
resumen = limpiador.ultima_ejecucion
mostrar_tabla(pd.DataFrame({
    'Etapa': ['Entrada', 'Salida'],
    'Filas': [resumen['filas_entrada'], resumen['filas_salida']],
    'Columnas': [resumen['columnas_entrada'], resumen['columnas_salida']],
}), 'Dimensiones antes y después de limpiar', {'Filas': '{:,.0f}', 'Columnas': '{:.0f}'})
print('Columnas leídas:', ', '.join(raw.columns))
print('Columnas excluidas por vacías:', ', '.join(resumen['columnas_excluidas']) or 'ninguna')

Etapa,Filas,Columnas
Entrada,44.226,7
Salida,44.226,10


Columnas leídas: NroLicitacion, TipoLicitacion, FechaPublicacion, FechaCierre, EstadoLicitacion, TamanoProveedor, ResultadoOferta
Columnas excluidas por vacías: ninguna


Se conservan las 44.226 filas. Entran las siete columnas leídas y salen diez: la limpieza convierte fechas y textos y añade tres variables derivadas. Las tres columnas completamente vacías del archivo ya no se leen, por lo que no queda ninguna por excluir; la regla que lo comprueba se sigue aplicando.

### 2.3. Verificación de calidad
El validador ejecuta reglas mediante una interfaz común: ese intercambio de reglas evidencia polimorfismo. La siguiente tabla contiene exclusivamente las reglas superadas.

Las reglas heredan de `ReglaValidacion` y el validador invoca el mismo método en cada una: permite añadir validaciones sin cambiar su coordinación. `None` selecciona las reglas predeterminadas; una colección vacía se rechaza. El limpiador devuelve copias y expone una copia de su resumen. Así, la lectura, las transformaciones y la validación mantienen responsabilidades distintas.

In [6]:
validacion = ValidadorDatasetProcesado().validar(datos, len(raw))
mostrar_tabla(pd.DataFrame({'Regla': validacion['detalle_reglas'], 'Estado': 'OK'}), 'Reglas de calidad superadas')

Regla,Estado
columnas_vacias_descartadas,OK
integridad_filas,OK
columnas_obligatorias,OK
resultado_oferta,OK
fechas,OK
variables_derivadas,OK


## 3. Contrato del cálculo y ejemplo manual
Se selecciona el estado `Adjudicada`. El denominador suma ganadoras y perdedoras por grupo; NA y resultados desconocidos se informan aparte. Un denominador cero produce NaN, mostrado como «—». Se rechazan grupos faltantes dentro del estado seleccionado y la categoría reservada `All`.

### 3.1. Datos del ejemplo
La oferta del grupo E está revocada y no participa del cálculo. El grupo B no tiene resultado válido.

In [7]:
from F3.test_algoritmos import muestra
mini = muestra()
mostrar_tabla(mini.rename(columns={'EstadoLicitacion': 'Estado', 'ResultadoOferta': 'Resultado'}), 'Ofertas del ejemplo manual')

Grupo,Estado,Resultado
A,Adjudicada,Ganadora
A,Adjudicada,Ganadora
A,Adjudicada,Perdedora
B,Adjudicada,—
C,Adjudicada,Perdedora
D,Adjudicada,Ganadora
E,Revocada,Ganadora


### 3.2. Resultado esperado
A tiene dos ganadoras de tres ofertas válidas: 66,67 %. C tiene 0 % y D tiene 100 %. El total es 3/5 = 60 %. Se suman los conteos; promediar porcentajes de grupos no produciría el mismo denominador.

In [8]:
resultado = tabla_proporciones_iterativa(mini, 'Grupo')
assert resultado.loc['All', '% Ganadora'] == 60
mostrar_proporciones(resultado, 'Proporciones del ejemplo')

Grupo,Ganadoras,Perdedoras,Sin resultado válido,Ofertas válidas,Ganadoras (%),Perdedoras (%)
A,2,1,0,3,"66,67","33,33"
B,0,0,1,0,—,—
C,0,1,0,1,"0,00","100,00"
D,1,0,0,1,"100,00","0,00"
Total,3,2,1,5,"60,00","40,00"


## 4. Alternativas y recursividad
### 4.1. Responsabilidades de cada alternativa

| Alternativa | Trabajo específico |
| --- | --- |
| Referencia F2 | Filtra resultados y agrupa varias veces; se conserva sin cambios. |
| Iterativa | Recorre registros y acumula tres contadores por grupo en un diccionario. |
| Recursiva | Divide intervalos, cuenta bloques y suma diccionarios parciales. |
| Agrupada | Agrupa una vez por categoría y código de resultado mediante pandas. |

Las tres alternativas nuevas comparten preparación y construcción de salida. La medición incluye esos pasos: no se excluyen conversiones que favorezcan a alguna alternativa. La agrupación utiliza `groupby` (The pandas development team, s. f.-a).

### 4.2. Descomposición recursiva
El caso base cuenta hasta 256 filas; un intervalo mayor se divide por el punto medio. Cada llamada reduce el rango y garantiza terminación. Se pasan posiciones, sin copiar mitades de DataFrames. La combinación suma conteos antes de calcular porcentajes.

La descomposición permite combinar resúmenes parciales, pero esta implementación es secuencial. El bloque 256 es configurable y no se presenta como óptimo.

In [9]:
import inspect
from src.analisis import _contar_dividiendo
print(inspect.getsource(_contar_dividiendo))

def _contar_dividiendo(grupos, codigos, inicio, fin, bloque_base):
    """Divide rangos, cuenta hojas y combina diccionarios en profundidad."""
    if fin - inicio <= bloque_base:
        return _contar_rango(grupos, codigos, inicio, fin)
    medio = (inicio + fin) // 2
    izquierda = _contar_dividiendo(grupos, codigos, inicio, medio, bloque_base)
    derecha = _contar_dividiendo(grupos, codigos, medio, fin, bloque_base)
    for grupo, valores in derecha.items():
        acumulado = izquierda.setdefault(grupo, [0, 0, 0])
        for i in range(3):
            acumulado[i] += valores[i]
    return izquierda



### 4.3. Comprobación de distintas particiones
Se compara el ejemplo con bloques de 1, 3 y 256. Los dos primeros fuerzan divisiones; el último cuenta directamente. La igualdad demuestra que la combinación conserva el resultado.

In [10]:
particiones = []
for bloque in (1, 3, 256):
    pd.testing.assert_frame_equal(tabla_proporciones_recursiva(mini, 'Grupo', bloque_base=bloque), resultado)
    particiones.append({'Bloque base': bloque, 'Equivalencia': 'OK'})
mostrar_tabla(pd.DataFrame(particiones), 'Verificación de particiones recursivas')

Bloque base,Equivalencia
1,OK
3,OK
256,OK


## 5. Pruebas y casos límite visibles
### 5.1. Suite reejecutable
Las ocho pruebas algorítmicas incluyen un oráculo manual, conservación de la entrada, índices repetidos, grupos inválidos, resultados desconocidos, vacíos, muestras aleatorias reproducibles y datos reales. Se ejecutan también las 14 pruebas POO y las siete de lectura: 29 pruebas en total. Incluyen la inmutabilidad del contrato, las reglas personalizadas, el rechazo de una lista de reglas vacía y la equivalencia entre opciones de lectura. El registro detallado queda en esta salida.

In [11]:
import unittest, io
suite = unittest.defaultTestLoader.loadTestsFromNames(['F3.test_algoritmos', 'F3.test_nucleo_poo', 'F3.test_lectura'])
log = io.StringIO()
pruebas = unittest.TextTestRunner(stream=log, verbosity=2).run(suite)
print(log.getvalue())
assert pruebas.wasSuccessful()

test_bloques_invalidos (F3.test_algoritmos.PruebasAlgoritmos.test_bloques_invalidos) ... ok
test_columnas_ausentes (F3.test_algoritmos.PruebasAlgoritmos.test_columnas_ausentes) ... ok
test_dataset_real_ambas_agrupaciones (F3.test_algoritmos.PruebasAlgoritmos.test_dataset_real_ambas_agrupaciones) ... ok
test_faltantes_resultados_e_indices_duplicados (F3.test_algoritmos.PruebasAlgoritmos.test_faltantes_resultados_e_indices_duplicados) ... ok
test_grupo_faltante_y_categoria_reservada (F3.test_algoritmos.PruebasAlgoritmos.test_grupo_faltante_y_categoria_reservada) ... ok
test_oraculo_manual_y_totales (F3.test_algoritmos.PruebasAlgoritmos.test_oraculo_manual_y_totales) ... ok
test_particiones_aleatorias_reproducibles (F3.test_algoritmos.PruebasAlgoritmos.test_particiones_aleatorias_reproducibles) ... ok
test_vacio_filtro_sin_filas_y_otro_estado (F3.test_algoritmos.PruebasAlgoritmos.test_vacio_filtro_sin_filas_y_otro_estado) ... ok
test_contrato_conserva_columnas_ante_mutacion_externa (F3.te

### 5.2. Grupo sin resultados válidos
Se conserva el conteo de resultados sin clasificar, pero el denominador válido es cero. Mostrar 0 % sugeriría una proporción calculable que aquí no existe.

In [12]:
sin_resultado = mini.loc[mini.Grupo.eq('B')]
tabla_sin_resultado = tabla_proporciones_agrupada(sin_resultado, 'Grupo')
assert tabla_sin_resultado.loc['B', 'All'] == 0
assert pd.isna(tabla_sin_resultado.loc['B', '% Ganadora'])
mostrar_proporciones(tabla_sin_resultado, 'Caso límite: sin resultados válidos')

Grupo,Ganadoras,Perdedoras,Sin resultado válido,Ofertas válidas,Ganadoras (%),Perdedoras (%)
B,0,0,1,0,—,—
Total,0,0,1,0,—,—


### 5.3. Estado sin registros
Si el estado seleccionado no aparece, la salida contiene solamente el total cero y porcentajes no definidos. Es distinto del caso anterior, donde sí había un registro con resultado desconocido.

In [13]:
sin_filas = tabla_proporciones_agrupada(mini, 'Grupo', estado='Desierta')
assert len(sin_filas) == 1 and sin_filas.loc['All', 'All'] == 0
mostrar_proporciones(sin_filas, 'Caso límite: ningún registro seleccionado')

Grupo,Ganadoras,Perdedoras,Sin resultado válido,Ofertas válidas,Ganadoras (%),Perdedoras (%)
Total,0,0,0,0,—,—


### 5.4. Entrada inválida
Los faltantes de grupo dentro del estado seleccionado detienen el cálculo, evitando perder categorías silenciosamente. La excepción se captura aquí únicamente para mostrar la evidencia.

La referencia F2 usa acceso por atributo a `ResultadoOferta` y produce `AttributeError` si falta esa columna; las nuevas variantes usan acceso por columna y producen `KeyError`. La suite comprueba esa diferencia para entradas inválidas sin modificar F2.

In [14]:
entrada_invalida = mini.copy()
entrada_invalida.loc[0, 'Grupo'] = None
try:
    tabla_proporciones_agrupada(entrada_invalida, 'Grupo')
except ValueError as error:
    print('Error esperado:', error)
else:
    raise AssertionError('La entrada inválida no fue rechazada')

Error esperado: Grupo: resolver faltantes antes de agrupar.


## 6. Resultados del dataset real
### 6.1. Equivalencia de algoritmos
Antes de interpretar resultados o comparar tiempos, se verifica que cada alternativa conserve exactamente la tabla de referencia para ambas variables.

In [15]:
equivalencias = []
for grupo in ('TamanoProveedor', 'TipoLicitacion'):
    esperado = tabla_proporciones(datos, grupo)
    for nombre, funcion in [('Iterativa', tabla_proporciones_iterativa), ('Recursiva', tabla_proporciones_recursiva), ('Agrupada', tabla_proporciones_agrupada)]:
        pd.testing.assert_frame_equal(funcion(datos, grupo), esperado)
        equivalencias.append({'Variable': NOMBRES_GRUPO[grupo], 'Algoritmo': nombre, 'Resultado': 'Igual a F2'})
mostrar_tabla(pd.DataFrame(equivalencias), 'Equivalencia sobre el dataset completo')

Variable,Algoritmo,Resultado
Tamaño del proveedor,Iterativa,Igual a F2
Tamaño del proveedor,Recursiva,Igual a F2
Tamaño del proveedor,Agrupada,Igual a F2
Tipo de licitación,Iterativa,Igual a F2
Tipo de licitación,Recursiva,Igual a F2
Tipo de licitación,Agrupada,Igual a F2


### 6.2. Proporciones por tamaño de proveedor
Cada porcentaje utiliza las ofertas válidas de su propio grupo. Las diferencias son descriptivas y no demuestran un efecto causal del tamaño de empresa.

In [16]:
mostrar_proporciones(tabla_proporciones_agrupada(datos, 'TamanoProveedor'), 'Ofertas ganadoras por tamaño del proveedor')

Tamaño del proveedor,Ganadoras,Perdedoras,Sin resultado válido,Ofertas válidas,Ganadoras (%),Perdedoras (%)
Grande,12.575,6.255,0,18.830,"66,78","33,22"
Mediana,4.982,4.001,0,8.983,"55,46","44,54"
Micro,1.680,1.554,0,3.234,"51,95","48,05"
NoClasificado,1.803,1.576,0,3.379,"53,36","46,64"
Pequeña,5.022,4.577,0,9.599,"52,32","47,68"
Total,26.062,17.963,0,44.025,"59,20","40,80"


### 6.3. Proporciones por tipo de licitación
Se presenta en otra tabla porque corresponde a una agrupación distinta. Los denominadores deben acompañar a los porcentajes: categorías con pocas ofertas no tienen el mismo respaldo que las más numerosas.

In [17]:
mostrar_proporciones(tabla_proporciones_agrupada(datos, 'TipoLicitacion'), 'Ofertas ganadoras por tipo de licitación')

Tipo de licitación,Ganadoras,Perdedoras,Sin resultado válido,Ofertas válidas,Ganadoras (%),Perdedoras (%)
Licitación Privada Mayor a 1000 UTM,62,10,0,72,"86,11","13,89"
Licitación Privada Mayor a 5000 (I2),1,1,0,2,"50,00","50,00"
Licitación Privada entre 100 y 1000 UTM.,11,1,0,12,"91,67","8,33"
Licitación Pública Entre 100 y 1000 UTM (LE),14.686,8.904,0,23.590,"62,26","37,74"
Licitación Pública Mayor 1000 UTM (LP),8.066,6.328,0,14.394,"56,04","43,96"
Licitación Pública Mayor a 5000 (LR),1.652,1.559,0,3.211,"51,45","48,55"
Licitación Pública Menor a 100 UTM (L1),1.584,1.160,0,2.744,"57,73","42,27"
Total,26.062,17.963,0,44.025,"59,20","40,80"


## 7. Diseño experimental y trazabilidad
### 7.1. Qué se mide
Se comparan cuatro tamaños: 100, 1.000, 10.000 y 44.226 filas. Las muestras son anidadas y provienen de una permutación con semilla 2026. Cada alternativa recibe el mismo DataFrame, obtenido con la lectura de columnas necesarias de la sección 2.1.

Se realizan siete rondas de tres ejecuciones, con verificación y calentamiento previos y orden de algoritmos alternado con semilla. Se incluye filtro, conversión, conteo y tabla de salida. Se excluyen lectura, limpieza, muestreo y comprobación de equivalencia. La lectura se mide por separado en la sección 12.

`timeit` desactiva GC durante cada ronda. Se guarda el mínimo como referencia de menor interferencia, mediana, cuartiles y observaciones individuales. Los cuartiles describen la sesión; no son intervalos de confianza (Python Software Foundation, s. f.-a).

La memoria se mide tres veces en ejecuciones separadas con `tracemalloc`. Su pico excluye la entrada preexistente y no equivale al RSS ni garantiza incluir toda la memoria nativa (Python Software Foundation, s. f.-b).

### 7.2. Responsabilidades del script de medición

| Función | Responsabilidad |
| --- | --- |
| `medir` | Preparar tamaños y variables del experimento. |
| `medir_caso` | Comprobar equivalencia y coordinar un caso. |
| `medir_tiempos` | Reutilizar temporizadores y alternar las rondas. |
| `medir_memoria` | Recoger picos en ejecuciones separadas. |
| `resumir_medicion` | Calcular estadísticas y conservar observaciones. |
| `ejecutar` | Cargar mediante interfaces existentes y guardar evidencia. |

El conteo de filas adjudicadas se calcula una vez por muestra y los cuartiles en una operación. Los bucles restantes representan casos y repeticiones del experimento, no cruces entre todas las filas. Eliminarlos reduciría la evidencia; no demostraría una mejora algorítmica.

### 7.3. Evidencia utilizada
Por defecto se muestran mediciones guardadas cuyos hashes se comprueban. Para medir de nuevo, activar `REGENERAR_MEDICIONES`; si cambia el código, deben regenerarse. La nueva ejecución puede producir tiempos distintos.

In [18]:
from F3.medir_algoritmos import ejecutar
REGENERAR_MEDICIONES = False  # True repite el experimento y guarda nueva evidencia.
evidencia = raiz / 'evidencias/F3_algoritmos/mediciones.json'
if REGENERAR_MEDICIONES or not evidencia.exists():
    ejecutar()
registro = json.loads(evidencia.read_text(encoding='utf-8'))
assert registro['raw_sha256'] == sha256_archivo(ruta)
assert registro['analisis_sha256'] == sha256_archivo(raiz / 'src/analisis.py')
assert registro['medicion_sha256'] == sha256_archivo(raiz / 'F3/medir_algoritmos.py')
print('Mediciones de:', registro['fecha_utc'])
print('Python:', registro['python'])
print('Plataforma:', registro['sistema'])
print('Repeticiones:', registro['repeticiones'], '| Ejecuciones por ronda:', registro['ejecuciones_por_repeticion'])
mediciones = pd.DataFrame(registro['mediciones'])
assert len(mediciones) == 32
print('Casos experimentales:', len(mediciones))

Mediciones de: 2026-09-28T01:09:36.831307+00:00
Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Plataforma: Windows-11-10.0.26200-SP0
Repeticiones: 7 | Ejecuciones por ronda: 3
Casos experimentales: 32


## 8. Rendimiento temporal
### 8.1. Crecimiento por tamaño de proveedor
La tabla contiene solo medianas en milisegundos. Separar memoria y dispersión facilita comparar cómo cambia el tiempo al aumentar las filas.

In [19]:
def mostrar_crecimiento(grupo):
    vista = mediciones.loc[mediciones.grupo.eq(grupo)].pivot(index='filas_entrada', columns='algoritmo', values='mediana_ms')
    vista = vista.rename(columns=NOMBRES_ALGORITMO).rename_axis('Filas').reset_index()
    vista.columns.name = None
    mostrar_tabla(vista, NOMBRES_GRUPO[grupo] + ' — medianas (ms)',
                  {'Filas': '{:,.0f}', **{nombre: '{:.3f}' for nombre in NOMBRES_ALGORITMO.values()}})
mostrar_crecimiento('TamanoProveedor')

Filas,Agrupada,Iterativa,Recursiva,Referencia F2
100,"3,054","2,197","2,176","4,366"
1.000,"3,184","2,358","2,357","4,321"
10.000,"5,920","6,219","6,426","7,374"
44.226,"19,024","23,082","22,277","22,118"


### 8.2. Crecimiento por tipo de licitación
Es la misma operación sobre otra distribución de categorías. Se muestran los resultados por separado para evitar repetir una columna de agrupación en todas las filas.

In [20]:
mostrar_crecimiento('TipoLicitacion')

Filas,Agrupada,Iterativa,Recursiva,Referencia F2
100,"2,952","1,993","1,965","3,968"
1.000,"3,377","2,404","2,417","4,346"
10.000,"7,240","6,747","7,516","9,224"
44.226,"20,964","23,056","22,924","24,444"


### 8.3. Interpretación del crecimiento
En muestras pequeñas, agrupar tiene un costo fijo que puede superar al recorrido con diccionario. En 100 filas, el bloque base 256 evita que la variante recursiva divida: diferencias pequeñas frente a la iterativa no son evidencia de una ventaja de la recursividad.

Al aumentar n, pesa más el procesamiento de registros. Con la lectura de columnas necesarias, cada filtro de la referencia copia solo diez columnas y sus recorridos adicionales cuestan poco: en el conjunto completo, la iterativa y la recursiva quedan a la par de la referencia, y solo la agrupada es claramente más rápida. La recursiva no ofrece una ventaja sostenida sobre la iterativa en las mediciones guardadas. Ninguna tabla aislada demuestra una ley asintótica.

### 8.4. Dispersión temporal: tamaño de proveedor
Se muestran mínimo, mediana y cuartiles para las 44.226 filas, con 44.025 seleccionadas por estado. Son resúmenes de esta sesión, no garantías de rendimiento.

In [21]:
completo = mediciones.loc[mediciones.filas_entrada.eq(len(datos))].copy()
def mostrar_tiempos_completos(grupo):
    tabla = completo.loc[completo.grupo.eq(grupo)]
    vista = pd.DataFrame({'Algoritmo': tabla.algoritmo.map(NOMBRES_ALGORITMO),
                          'Mínimo (ms)': tabla.min_ms, 'Mediana (ms)': tabla.mediana_ms,
                          'Q25 (ms)': tabla.q25_ms, 'Q75 (ms)': tabla.q75_ms})
    mostrar_tabla(vista, NOMBRES_GRUPO[grupo] + ' — dispersión temporal',
                  {col: '{:.3f}' for col in vista.columns if col != 'Algoritmo'})
mostrar_tiempos_completos('TamanoProveedor')

Algoritmo,Mínimo (ms),Mediana (ms),Q25 (ms),Q75 (ms)
Referencia F2,"20,764","22,118","21,263","22,812"
Iterativa,"22,088","23,082","22,456","23,473"
Recursiva,"21,505","22,277","22,100","23,308"
Agrupada,"18,168","19,024","18,318","19,777"


### 8.5. Dispersión temporal: tipo de licitación
La agrupada también obtiene el menor mínimo. Por tipo, su ventaja sobre la iterativa y la recursiva es cercana a un 9 %: debe interpretarse con cautela y junto con las otras métricas.

In [22]:
mostrar_tiempos_completos('TipoLicitacion')

Algoritmo,Mínimo (ms),Mediana (ms),Q25 (ms),Q75 (ms)
Referencia F2,"23,555","24,444","24,236","26,895"
Iterativa,"22,649","23,056","22,916","23,360"
Recursiva,"22,422","22,924","22,806","23,891"
Agrupada,"20,177","20,964","20,595","21,791"


## 9. Memoria auxiliar observada
La siguiente tabla contiene únicamente la mediana de los tres picos trazados para el conjunto completo, expresada en MiB (2²⁰ bytes). No mide la memoria total del CSV ni de todo el proceso.

In [23]:
memoria = completo.pivot(index='algoritmo', columns='grupo', values='pico_trazado_bytes_mediana') / 1024**2
memoria = memoria.rename(index=NOMBRES_ALGORITMO, columns=NOMBRES_GRUPO)
memoria.columns.name = None
memoria = memoria.rename_axis('Algoritmo').reset_index()
mostrar_tabla(memoria, 'Pico de memoria trazada — mediana (MiB)', {nombre: '{:.2f}' for nombre in NOMBRES_GRUPO.values()})

Algoritmo,Tamaño del proveedor,Tipo de licitación
Agrupada,"4,55","4,55"
Iterativa,"4,55","4,55"
Recursiva,"4,55","4,55"
Referencia F2,"6,51","6,51"


Las alternativas nuevas presentan picos muy similares entre sí y menores que la referencia. Esto sugiere que la preparación compartida pesa más que sus diferencias de conteo, pero no constituye una atribución causal del uso de memoria por función.

El filtrado copia todas las columnas recibidas antes de extraer las necesarias. Leer solo las columnas del análisis (sección 2.1) reduce ese costo: en la medición anterior, con las 74 columnas, el pico era de 40,81 MiB en la referencia y de 26,07 MiB en las alternativas nuevas. La recursividad conserva diccionarios parciales, por lo que su costo espacial no se reduce a la pila de llamadas.

## 10. Complejidad temporal y espacial

Sea N el total de filas recibidas, n las seleccionadas, g los grupos y b el bloque base; el número de columnas se considera fijo. El filtro cuesta O(N), la preparación O(n), y ordenar categorías O(g log g).

| Alternativa | Conteo y combinación (costo esperado de hashing) | Memoria auxiliar del conteo |
| --- | --- | --- |
| Iterativa | O(n) | O(g) |
| Recursiva | O(n + S), donde S suma las entradas combinadas en nodos internos | Diccionarios parciales y pila; cota O(n + log L) |
| Agrupada | O(n + g), considerando tres resultados posibles | O(n + g) para estructuras de agrupación |
| Referencia F2 | Varios recorridos y agrupaciones; con tres resultados fijos, orden esperado lineal más ordenamiento | O(n + g) |

L = max(1, ceil(n/b)) aproxima la cantidad de hojas. La profundidad recursiva es O(log L); S está acotado por O(gL) y por O(n log L). Con g y b fijos, el conteo recursivo sigue siendo lineal; con muchos grupos diferentes puede acercarse a O(n log L). Los diccionarios de subárboles ya resueltos se conservan mientras se calcula la otra mitad: la memoria no se reduce a la pila.

Todas las funciones completas materializan datos filtrados y salida: la memoria auxiliar total es O(N + n + g), bajo columnas fijas, incluyendo la máscara. La recursividad no elimina esas asignaciones. Cambiar constantes puede mejorar los tiempos sin cambiar el orden asintótico.

## 11. Elección para F3 y alcance de la mejora
Se selecciona la variante agrupada para el análisis del conjunto completo: obtuvo la menor mediana en ambas variables y un pico trazado comparable con las otras alternativas nuevas. La referencia F2 permanece intacta por compatibilidad. La siguiente tabla calcula la razón entre tiempos; no representa una aceleración de toda la aplicación.

In [24]:
decisiones = []
for grupo in ('TamanoProveedor', 'TipoLicitacion'):
    tabla = completo.loc[completo.grupo.eq(grupo)].set_index('algoritmo')
    decisiones.append({'Variable': NOMBRES_GRUPO[grupo],
        'F2 (ms)': tabla.loc['referencia_f2', 'mediana_ms'],
        'Agrupada (ms)': tabla.loc['agrupada', 'mediana_ms'],
        'Razón F2 / agrupada': tabla.loc['referencia_f2', 'mediana_ms'] / tabla.loc['agrupada', 'mediana_ms']})
mostrar_tabla(pd.DataFrame(decisiones), 'Comparación que respalda la elección',
              {'F2 (ms)': '{:.3f}', 'Agrupada (ms)': '{:.3f}', 'Razón F2 / agrupada': '{:.2f}'})

Variable,F2 (ms),Agrupada (ms),Razón F2 / agrupada
Tamaño del proveedor,"22,118","19,024","1,16"
Tipo de licitación,"24,444","20,964","1,17"


La evidencia guardada muestra una razón cercana a 1,16 por tamaño y 1,17 por tipo. Es menor que en la medición anterior con las 74 columnas (1,74 y 1,72): gran parte del costo de la referencia provenía de filtrar columnas que el cálculo no usa, y leer solo las necesarias redujo el tiempo de todas las alternativas. La agrupada sigue siendo la más rápida en ambas variables, entre un 9 % y un 18 % por debajo de la iterativa y la recursiva. Los datos no demuestran que pandas sea siempre más rápido ni justifican un selector automático según tamaño.

La recursividad demuestra descomposición, terminación y combinación correcta en un problema real. Se conserva como alternativa evaluada, aunque no resulte elegida para el análisis habitual. La selección se limita al volumen, cardinalidad, equipo y versiones registrados. Si se regeneran las mediciones, deben revisarse también estas conclusiones.

## 12. Rendimiento de la lectura de datos
Esta sección compara tres formas de cargar el CSV de licitaciones con `LectorCSV`. Mide tiempo y memoria, comprueba que los datos necesarios no cambian y justifica qué opción conviene. La opción «actual» es la lectura completa, que siguen usando F1 y F2; el análisis de F3 usa la opción con columnas y tipos (sección 2.1).

**Recorrido:** opciones y columnas → herramientas → equivalencia → evidencia → tiempos → crecimiento → memoria → flujo completo → decisión.

### 12.1. Importaciones y presentación
Se reutilizan el entorno y `mostrar_tabla` de la sección 1. Los nombres de esta sección llevan el sufijo `_lectura` cuando coinciden con variables de las mediciones de algoritmos.

In [25]:
from src.datos import COLUMNAS_ANALISIS, TIPOS_ANALISIS
from F3.medir_lectura import RUTA, SHA256_CSV, crear_lectores, comprobar_equivalencia, sha256_codigo
NOMBRES_OPCION = {'actual': 'Actual (todas las columnas)', 'solo_columnas': 'Solo columnas necesarias',
                  'columnas_y_tipos': 'Columnas y tipos definidos'}
MIB = 1024 ** 2

### 12.2. Opciones comparadas y columnas acordadas
El CSV tiene 44.226 filas y 74 columnas (~72 MB). F1 y F2 exploran todas las columnas; el análisis de F3 solo necesita algunas.

| Opción | Configuración de `LectorCSV` |
| --- | --- |
| Actual | Sin cambios: lee las 74 columnas y pandas infiere los tipos. |
| Solo columnas necesarias | `columnas=COLUMNAS_ANALISIS` (`usecols` de pandas). |
| Columnas y tipos definidos | Además `tipos=TIPOS_ANALISIS` (`dtype`): `category` para las cuatro categóricas. |

Las columnas se acordaron según lo que exige el código existente, no solo el cálculo:

In [26]:
motivo = {
    'NroLicitacion': 'Columna clave exigida por LimpiadorLicitaciones',
    'TipoLicitacion': 'Agrupación de proporciones',
    'TamanoProveedor': 'Agrupación de proporciones',
    'ResultadoOferta': 'Numerador y denominador de las proporciones',
    'EstadoLicitacion': 'Filtro de licitaciones adjudicadas',
    'FechaPublicacion': 'ReglaFechas y plazo_cierre_dias',
    'FechaCierre': 'ReglaFechas y plazo_cierre_dias',
}
mostrar_tabla(pd.DataFrame({'Columna': COLUMNAS_ANALISIS,
                            'Tipo al leer': [TIPOS_ANALISIS.get(c, 'inferido') for c in COLUMNAS_ANALISIS],
                            'Motivo': [motivo[c] for c in COLUMNAS_ANALISIS]}), 'Columnas leídas por las opciones nuevas')

Columna,Tipo al leer,Motivo
NroLicitacion,inferido,Columna clave exigida por LimpiadorLicitaciones
TipoLicitacion,category,Agrupación de proporciones
TamanoProveedor,category,Agrupación de proporciones
ResultadoOferta,category,Numerador y denominador de las proporciones
EstadoLicitacion,category,Filtro de licitaciones adjudicadas
FechaPublicacion,inferido,ReglaFechas y plazo_cierre_dias
FechaCierre,inferido,ReglaFechas y plazo_cierre_dias


Las fechas no se tipan al leer: la limpieza ya las convierte con formato mixto y excluye el año 1900; repetir esa lógica en la lectura la duplicaría. `NroLicitacion` conserva el tipo inferido porque la limpieza no lo normaliza y así los datos limpios quedan idénticos a la lectura actual.

### 12.3. Herramientas de medición

| Herramienta | Qué mide | Uso aquí |
| --- | --- | --- |
| `timeit` (basado en `perf_counter`) | Tiempo transcurrido, con GC desactivado | 7 rondas por opción, orden alternado con semilla 2026 |
| `DataFrame.memory_usage(deep=True)` | Tamaño del DataFrame resultante, incluyendo textos | Métrica principal de memoria |
| `tracemalloc` | Pico de asignaciones rastreadas por Python | 3 ejecuciones separadas del tiempo |

Se reutilizan `medir_tiempos`, `medir_memoria` y `resumir_medicion` de `medir_algoritmos.py`, para que ambos experimentos sigan la misma metodología. `tracemalloc` no equivale a la memoria total del proceso ni ve todos los búferes internos del lector de pandas (Python Software Foundation, s. f.-b). Por eso el tamaño del DataFrame es la referencia principal.

### 12.4. Equivalencia antes de medir
Una lectura más rápida que entrega otros datos no es una optimización. Cada opción se lee, se limpia y se valida. Luego se comparan con la lectura actual las siete columnas acordadas, las tres variables derivadas y las tablas de proporciones por tamaño y tipo. La comparación se hace después de limpiar: la limpieza convierte los textos a `string`, y la opción con tipos entrega `category` al leer.

In [27]:
equivalencia = comprobar_equivalencia(crear_lectores(), RUTA)
vista = pd.DataFrame(equivalencia)
vista['opcion'] = vista.opcion.map(NOMBRES_OPCION)
mostrar_tabla(vista.rename(columns={'opcion': 'Opción', 'filas': 'Filas', 'columnas_limpias': 'Columnas tras limpiar',
                                    'columnas_necesarias': 'Columnas necesarias', 'validacion': 'Validación',
                                    'proporciones': 'Proporciones'}),
              'Equivalencia con la lectura actual', {'Filas': '{:,.0f}'})

Opción,Filas,Columnas tras limpiar,Columnas necesarias,Validación,Proporciones
Actual (todas las columnas),44.226,74,iguales,OK,iguales
Solo columnas necesarias,44.226,10,iguales,OK,iguales
Columnas y tipos definidos,44.226,10,iguales,OK,iguales


### 12.5. Evidencia utilizada
Por defecto se muestran las mediciones guardadas. Antes se comprueban los hashes del CSV, de `src/datos.py` y de los scripts de medición. En los archivos de código, el hash se calcula con saltos de línea normalizados: git en Windows los convierte a CRLF, y sin normalizar la misma versión tendría otro hash en cada equipo. Para medir de nuevo, activar `REGENERAR_MEDICIONES_LECTURA`; si cambia alguno de esos archivos, es obligatorio hacerlo. Una nueva ejecución produce tiempos algo distintos.

In [28]:
from F3.medir_lectura import ejecutar as ejecutar_lectura
REGENERAR_MEDICIONES_LECTURA = False  # True repite el experimento y guarda nueva evidencia.
evidencia_lectura = raiz / 'F3/mediciones_lectura/mediciones.json'
if REGENERAR_MEDICIONES_LECTURA or not evidencia_lectura.exists():
    ejecutar_lectura()
registro_lectura = json.loads(evidencia_lectura.read_text(encoding='utf-8'))
assert registro_lectura['raw_sha256'] == sha256_archivo(RUTA) == SHA256_CSV
# Código: huella con saltos de línea normalizados, igual en Windows y en otros sistemas.
assert registro_lectura['datos_sha256'] == sha256_codigo(raiz / 'src/datos.py')
assert registro_lectura['medicion_sha256'] == sha256_codigo(raiz / 'F3/medir_lectura.py')
assert registro_lectura['medir_algoritmos_sha256'] == sha256_codigo(raiz / 'F3/medir_algoritmos.py')
print('Mediciones de:', registro_lectura['fecha_utc'])
print('Python:', registro_lectura['python'])
print('Plataforma:', registro_lectura['sistema'])
print('Repeticiones:', registro_lectura['repeticiones'], '| Ejecuciones por ronda:', registro_lectura['ejecuciones_por_repeticion'])
mediciones_lectura = pd.DataFrame(registro_lectura['mediciones'])
mediciones_lectura['Opción'] = mediciones_lectura.opcion.map(NOMBRES_OPCION)
lectura = mediciones_lectura.loc[mediciones_lectura.etapa.eq('lectura')]
completo_lectura = lectura.loc[lectura.filas.eq(lectura.filas.max())].set_index('opcion')
flujo = mediciones_lectura.loc[mediciones_lectura.etapa.eq('flujo_completo')].set_index('opcion')
print('Casos experimentales:', len(mediciones_lectura))

Mediciones de: 2026-09-28T01:10:31.196633+00:00
Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Plataforma: Windows-11-10.0.26200-SP0
Repeticiones: 7 | Ejecuciones por ronda: 1
Casos experimentales: 12


### 12.6. Tiempo de lectura del archivo completo
Se informan el mínimo (la ejecución con menos interferencia), la mediana y los cuartiles de siete rondas. Los cuartiles describen esta sesión; no son intervalos de confianza (Python Software Foundation, s. f.-a).

In [29]:
vista = pd.DataFrame({'Opción': completo_lectura['Opción'], 'Columnas': completo_lectura.columnas,
                      'Mínimo (ms)': completo_lectura.min_ms, 'Mediana (ms)': completo_lectura.mediana_ms,
                      'Q25 (ms)': completo_lectura.q25_ms, 'Q75 (ms)': completo_lectura.q75_ms,
                      'Reducción de mediana (%)': (1 - completo_lectura.mediana_ms / completo_lectura.loc['actual', 'mediana_ms']) * 100})
mostrar_tabla(vista, 'Lectura de 44.226 filas — tiempos',
              {'Columnas': '{:.0f}', **{c: '{:.1f}' for c in vista.columns if c not in ('Opción', 'Columnas')}})

Opción,Columnas,Mínimo (ms),Mediana (ms),Q25 (ms),Q75 (ms),Reducción de mediana (%)
Actual (todas las columnas),74,"823,0","841,1","837,0","854,0","0,0"
Solo columnas necesarias,7,"398,4","411,2","404,3","421,7","51,1"
Columnas y tipos definidos,7,"404,1","412,3","408,1","428,8","51,0"


Leer 7 de 74 columnas reduce el tiempo aproximadamente a la mitad, no a una décima parte. `usecols` evita crear las columnas descartadas, pero el lector igual debe recorrer y separar cada línea completa del archivo. Definir `category` no acelera la lectura: la diferencia con "solo columnas" está dentro de la dispersión entre rondas.

### 12.7. Cómo crece el tiempo con las filas
Una sola medición no muestra la complejidad; hay que ver cómo cambia el tiempo al aumentar el volumen.

In [30]:
vista = lectura.pivot(index='filas', columns='opcion', values='mediana_ms')[list(NOMBRES_OPCION)]
vista['Razón actual / solo columnas'] = vista['actual'] / vista['solo_columnas']
vista = vista.rename(columns=NOMBRES_OPCION).rename_axis('Filas').reset_index()
vista.columns.name = None
mostrar_tabla(vista, 'Medianas de lectura por tamaño (ms)',
              {'Filas': '{:,.0f}', **{c: '{:.1f}' for c in NOMBRES_OPCION.values()}, 'Razón actual / solo columnas': '{:.2f}'})

Filas,Actual (todas las columnas),Solo columnas necesarias,Columnas y tipos definidos,Razón actual / solo columnas
1.000,"18,0","11,3","11,9","1,59"
10.000,"144,1","84,4","84,8","1,71"
44.226,"841,1","411,2","412,3","2,05"


Las tres opciones crecen de forma aproximadamente proporcional a las filas: leer es O(N·C), con N filas y C columnas recorridas por el lector. La ventaja de leer menos columnas aumenta levemente con el volumen: en muestras pequeñas pesa el costo fijo de abrir el archivo e interpretar el encabezado.

### 12.8. Memoria
El tamaño del DataFrame es la diferencia más grande entre las opciones.

In [31]:
vista = pd.DataFrame({'Opción': completo_lectura['Opción'],
                      'DataFrame (MiB)': completo_lectura.memoria_dataframe_bytes / MIB,
                      'Pico tracemalloc (MiB)': completo_lectura.pico_trazado_bytes_mediana / MIB,
                      'Reducción del DataFrame (%)': (1 - completo_lectura.memoria_dataframe_bytes / completo_lectura.loc['actual', 'memoria_dataframe_bytes']) * 100})
mostrar_tabla(vista, 'Lectura de 44.226 filas — memoria', {c: '{:.1f}' for c in vista.columns if c != 'Opción'})

Opción,DataFrame (MiB),Pico tracemalloc (MiB),Reducción del DataFrame (%)
Actual (todas las columnas),"195,7","43,5","0,0"
Solo columnas necesarias,"20,6","4,4","89,5"
Columnas y tipos definidos,"8,9","3,2","95,5"


Leer solo las columnas necesarias reduce el DataFrame en torno a un 89 %. Con tipos `category` baja aún más, porque cada categoría repetida se guarda una vez y las filas almacenan códigos enteros en lugar de textos. Aquí está el intercambio: los tipos no cambian el tiempo, pero sí la memoria.

### 12.9. Flujo completo: lectura, limpieza y proporciones
Como el costo de construir un índice, el ahorro al leer solo importa si se mantiene en el uso real. Se mide la lectura, `LimpiadorLicitaciones` y las dos tablas de proporciones.

In [32]:
vista = pd.DataFrame({'Opción': flujo['Opción'], 'Mínimo (ms)': flujo.min_ms, 'Mediana (ms)': flujo.mediana_ms,
                      'DataFrame limpio (MiB)': flujo.memoria_dataframe_bytes / MIB,
                      'Razón actual / opción': flujo.loc['actual', 'mediana_ms'] / flujo.mediana_ms})
mostrar_tabla(vista, 'Flujo completo — 44.226 filas',
              {'Mínimo (ms)': '{:.1f}', 'Mediana (ms)': '{:.1f}', 'DataFrame limpio (MiB)': '{:.1f}', 'Razón actual / opción': '{:.2f}'})

Opción,Mínimo (ms),Mediana (ms),DataFrame limpio (MiB),Razón actual / opción
Actual (todas las columnas),"1180,4","1199,2","171,5","1,00"
Solo columnas necesarias,"484,2","500,5","17,6","2,40"
Columnas y tipos definidos,"487,2","505,9","17,6","2,37"


El ahorro crece en el flujo completo: la limpieza copia y transforma todas las columnas que recibe, así que menos columnas también la abaratan. En cambio, la ventaja de memoria de `category` desaparece tras limpiar: `normalizar_categorias` convierte esas columnas a `string`, y ambas opciones nuevas terminan con el mismo DataFrame limpio.

### 12.10. Decisión sobre la lectura
**El análisis de F3 lee con `columnas=COLUMNAS_ANALISIS` y `tipos=TIPOS_ANALISIS` (sección 2.1).**

- **Tiempo:** la lectura del archivo completo baja de ~841 ms a ~412 ms en mediana, y el flujo completo de ~1.199 ms a ~506 ms (unas 2,4 veces más rápido). Además, el cálculo de proporciones de las secciones 8 y 11 se abarata: la referencia F2 pasa de ~67 ms con 74 columnas a ~22 ms.
- **Memoria:** el DataFrame leído pasa de ~196 MiB a ~9 MiB, y el limpio de ~172 MiB a ~18 MiB.
- **Equivalencia:** las columnas necesarias, la validación y las proporciones quedan idénticas a la lectura actual.
- **Por qué tipos y no solo columnas:** definir tipos reduce a menos de la mitad la memoria de la lectura respecto de seleccionar solo columnas. En esta sesión, su mediana de lectura es prácticamente igual (412 frente a 411 ms); se acepta esa diferencia a cambio del ahorro de memoria. El beneficio se pierde en la limpieza actual; conservar `category` durante la limpieza queda como mejora posible, no realizada.
- **F1 y F2** mantienen la lectura completa, que sigue siendo el comportamiento por defecto de `LectorCSV`, porque exploran todas las columnas.

La medición cambia una decisión: la lectura deja de ser igual para todas las fases y pasa a seleccionarse según lo que cada análisis necesita. F3 aplica la opción elegida desde la sección 2.1.

**Limitaciones:** las cifras corresponden a un equipo, una sesión y el sistema de archivos con caché después del calentamiento; una primera lectura en frío puede tardar más. `tracemalloc` no mide la memoria total del proceso.

## 13. Conclusión general de F3
- **Proporciones:** se usa la variante agrupada para el análisis del conjunto completo; con la evidencia guardada es unas 1,16 veces más rápida que la referencia F2 en ambas variables y la más rápida de las cuatro (sección 11). La función de F2 se conserva sin cambios.
- **Lectura:** el análisis de F3 lee solo las columnas necesarias con tipos definidos; el flujo de lectura, limpieza y proporciones es unas 2,4 veces más rápido y el DataFrame leído ocupa cerca de un 95 % menos memoria (sección 12.10). También reduce el tiempo de todos los algoritmos: la referencia F2 pasó de ~67 a ~22 ms en el conjunto completo. F1 y F2 mantienen la lectura completa.
- **Resultados:** ninguna de las dos mejoras cambia los resultados: las alternativas y las opciones de lectura se comprueban contra la referencia antes de medir.

Ambas decisiones se limitan al volumen, equipo y versiones registrados en cada evidencia.

## 14. Reproducibilidad y cierre

Desde la raíz del repositorio:

```powershell
.\.venv\Scripts\python.exe -m unittest F3.test_algoritmos F3.test_nucleo_poo F3.test_lectura -v
.\.venv\Scripts\python.exe F3\medir_algoritmos.py
.\.venv\Scripts\python.exe F3\medir_lectura.py
.\.venv\Scripts\python.exe F3\verificar_algoritmos.py
```

`medir_algoritmos.py` conserva tiempos individuales, picos, parámetros, versiones y hashes en `evidencias/F3_algoritmos/`; `medir_lectura.py` hace lo mismo para la lectura en `F3/mediciones_lectura/`. El verificador ejecuta este notebook en un kernel nuevo y registra su hash. El informe `docs/F3_APORTE_ALGORITMOS.md` sintetiza las decisiones y resultados de algoritmos expuestos aquí; no contiene la única explicación de la selección. Si se regeneran las mediciones, deben revisarse las cifras de las secciones 8.3, 8.5, 9, 11, 12.10 y 13.

Los aportes de lectura, limpieza, validación y algoritmos están integrados. Quedan pendientes las fuentes docentes y académicas, el cierre del informe institucional y la revisión de su PDF. La correspondencia final entre el informe y este notebook debe comprobarse al cerrar ese documento.

## 15. Referencias técnicas
- Python Software Foundation. (s. f.-a). *timeit — Measure execution time of small code snippets*. https://docs.python.org/3/library/timeit.html
- Python Software Foundation. (s. f.-b). *tracemalloc — Trace memory allocations*. https://docs.python.org/3/library/tracemalloc.html
- The pandas development team. (s. f.-a). *pandas.DataFrame.groupby*. https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html
- The pandas development team. (s. f.-b). *pandas.read_csv*. https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
- The pandas development team. (s. f.-c). *Categorical data*. https://pandas.pydata.org/docs/user_guide/categorical.html

Estas referencias respaldan el aporte técnico; la bibliografía grupal se completará durante la integración.